Add Population Density and Dominant Heating Type to Comparison File.

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from shapely.geometry import box, mapping
from pathlib import Path
from datetime import datetime


# ── Paths ──────────────────────────────────────────────────────────────────
CENSUS_RASTER = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\processed_datasets\Census\Census2022_100m\population.tif"
# Multi-band raster: 1 band per heating category (see HEAT_CATEGORIES below)
HEAT_RASTER   = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\processed_datasets\Census\Census2022_100m\dwelling_heat_src.tif"

INPUT_FILE  = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure_Differences.gpkg"
OUTPUT_FILE = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure_Differences_Census.gpkg"

# Band index (1-based, matches rasterio band numbering) -> category name
HEAT_CATEGORIES = {
    1: 'Gas',
    2: 'Heating_oil',
    3: 'Wood_pallets',
    4: 'Biomass_Non_wood_Biogas',
    5: 'Solar_geothermal_heat_pump',
    6: 'Electric_heating_no_heat_pumps',
    7: 'Coal',
    8: 'District_Heating',
    9: 'No_heating',
}


# ── Reprojection helper (shared by census + heat raster) ───────────────────
def reproject_raster_if_needed(raster_path, target_crs, label="", resampling=Resampling.nearest):
    """
    Reproject raster_path to target_crs if the CRS doesn't already match.
    Uses nearest-neighbour resampling by default. For the heat raster this is
    important: even though the values represent counts (not class codes),
    nearest-neighbour avoids interpolating/mixing counts across pixels during
    reprojection, which would distort per-pixel totals. Also fine for the
    census raster since that's what the original workflow used.
    Works for multi-band rasters (reprojects every band).
    Returns (path_to_use: Path, nodata_value).
    """
    with rasterio.open(raster_path) as src:
        orig_nodata = src.nodata

        if src.crs == target_crs:
            print(f"  ✓ {label} CRS match — no reprojection needed")
            return Path(raster_path), orig_nodata

        print(f"  ⚠️ {label} CRS mismatch ({src.crs} ≠ {target_crs}) — reprojecting...")

        reproj_path = Path(raster_path).with_name(
            f"{Path(raster_path).stem}_reprojected_{target_crs.to_epsg()}.tif"
        )
        reproj_path.parent.mkdir(parents=True, exist_ok=True)

        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds
        )

        out_nodata = orig_nodata if orig_nodata is not None else -9999

        kwargs = src.meta.copy()
        kwargs.update({
            "driver": "GTiff",
            "crs": target_crs,
            "transform": transform,
            "width": width,
            "height": height,
            "nodata": out_nodata,
            "compress": "lzw"
        })

        print(f"  Creating reprojected raster:\n  → {reproj_path}")

        with rasterio.open(reproj_path, "w", **kwargs) as dst:
            for band_index in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, band_index),
                    destination=rasterio.band(dst, band_index),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=orig_nodata,
                    dst_transform=transform,
                    dst_crs=target_crs,
                    dst_nodata=out_nodata,
                    resampling=resampling
                )

        print(f"  ✓ Reprojected raster created ({label})")
        return reproj_path, out_nodata


# ── Core function: population sum (fractional pixel weighting) ─────────────
def calculate_population_for_polygon(polygon_geom, raster_path, nodata_val, verbose=False):
    """
    Sum population pixels within *polygon_geom*, weighting edge pixels by the
    fraction of the pixel area that actually overlaps the polygon.
    """
    try:
        with rasterio.open(raster_path) as src:
            try:
                out_image, out_transform = rio_mask(
                    src,
                    [mapping(polygon_geom)],
                    crop=True,
                    all_touched=True,
                    filled=True,
                    nodata=nodata_val if nodata_val is not None else -9999
                )
            except ValueError as ve:
                if verbose:
                    print(f"      ⚠️  rio_mask failed: {ve}")
                return None

            data = out_image[0].astype(float)

            pixel_w    = abs(out_transform.a)
            pixel_h    = abs(out_transform.e)
            pixel_area = pixel_w * pixel_h

            total_pop = 0.0

            for r, c in np.ndindex(data.shape):
                val = data[r, c]

                if nodata_val is not None and val == nodata_val:
                    continue
                if np.isnan(val):
                    continue
                if val <= 0:
                    continue

                x_left  = out_transform.c + c * out_transform.a
                y_top   = out_transform.f + r * out_transform.e
                x_right = x_left + out_transform.a
                y_bot   = y_top  + out_transform.e

                pixel_box = box(
                    min(x_left, x_right), min(y_top, y_bot),
                    max(x_left, x_right), max(y_top, y_bot)
                )

                intersection = polygon_geom.intersection(pixel_box)
                if intersection.is_empty:
                    continue

                fraction   = intersection.area / pixel_area
                total_pop += val * fraction

            return round(total_pop, 2)

    except Exception as e:
        if verbose:
            print(f"      ⚠️  Unexpected error: {e}")
        return None


# ── New: multi-band weighted sums (one band per heating category) ──────────
def calculate_building_counts_by_band(polygon_geom, raster_path, nodata_val,
                                       categories_map, verbose=False):
    """
    For a MULTI-BAND raster where each band represents a heating category
    (band 1 = Gas count per pixel, band 2 = Heating_oil count per pixel, ...),
    compute the fractional pixel-overlap weighted SUM of counts for each band
    within polygon_geom. Uses the same intersection-area weighting as the
    population function, so partially covered edge pixels contribute
    proportionally instead of being fully counted or fully dropped.

    The pixel geometry (and therefore the per-pixel area fractions) is
    identical across all bands, so the fractions are computed only once and
    reused for every band — this avoids recomputing 9x the same shapely
    intersections.

    Returns a dict {category_name: weighted_sum} or None on failure.
    """
    try:
        with rasterio.open(raster_path) as src:
            n_bands = src.count
            if n_bands != len(categories_map):
                raise ValueError(
                    f"Raster has {n_bands} bands, but {len(categories_map)} "
                    f"categories are defined in HEAT_CATEGORIES."
                )

            try:
                out_image, out_transform = rio_mask(
                    src,
                    [mapping(polygon_geom)],
                    crop=True,
                    all_touched=True,
                    filled=True,
                    nodata=nodata_val if nodata_val is not None else -9999
                )
            except ValueError as ve:
                if verbose:
                    print(f"      ⚠️  rio_mask failed: {ve}")
                return None

            n_rows, n_cols = out_image.shape[1], out_image.shape[2]

            pixel_w    = abs(out_transform.a)
            pixel_h    = abs(out_transform.e)
            pixel_area = pixel_w * pixel_h

            # ── compute intersection-area fractions once (same for all bands) ──
            fractions = np.zeros((n_rows, n_cols), dtype=float)

            for r, c in np.ndindex((n_rows, n_cols)):
                x_left  = out_transform.c + c * out_transform.a
                y_top   = out_transform.f + r * out_transform.e
                x_right = x_left + out_transform.a
                y_bot   = y_top  + out_transform.e

                pixel_box = box(
                    min(x_left, x_right), min(y_top, y_bot),
                    max(x_left, x_right), max(y_top, y_bot)
                )

                intersection = polygon_geom.intersection(pixel_box)
                if intersection.is_empty:
                    continue

                fractions[r, c] = intersection.area / pixel_area

            if not np.any(fractions > 0):
                # polygon doesn't actually overlap any pixel area (can happen
                # for slivers after crop) -> nothing to sum
                return {cat_name: 0.0 for cat_name in categories_map.values()}

            # ── per-band weighted sum ───────────────────────────────────────
            band_sums = {}

            for band_pos, (band_idx_1based, cat_name) in enumerate(categories_map.items()):
                band_data = out_image[band_pos].astype(float)

                mask_valid = np.ones_like(band_data, dtype=bool)
                if nodata_val is not None:
                    mask_valid &= (band_data != nodata_val)
                mask_valid &= ~np.isnan(band_data)
                mask_valid &= (band_data > 0)
                mask_valid &= (fractions > 0)

                weighted_sum = float(np.sum(band_data[mask_valid] * fractions[mask_valid]))
                band_sums[cat_name] = round(weighted_sum, 4)

            return band_sums

    except Exception as e:
        if verbose:
            print(f"      ⚠️  Unexpected error: {e}")
        return None


def get_dominant_heat_type(band_sums, categories_map, decimals=4):
    """
    Return the dominant heating category (or categories, if tied) from a
    dict of {category_name: weighted_sum}.

    If several categories share the same maximum weighted sum, all tied
    category names are retained, ordered by their band/category code
    (1..9 as defined in HEAT_CATEGORIES) and joined with ";".

    Example:
        "Gas;Heating_oil"
    """
    if not band_sums:
        return None

    rounded = {cat: round(float(val), decimals) for cat, val in band_sums.items()}
    max_value = max(rounded.values())

    if max_value <= 0:
        return None

    dominant_names = {cat for cat, val in rounded.items() if val == max_value}

    # order by category code (1..9), i.e. the order of HEAT_CATEGORIES.values()
    ordered = [name for name in categories_map.values() if name in dominant_names]

    return ";".join(ordered)


# ── File checks ────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("POPULATION DENSITY + HEATING TYPE BREAKDOWN (9-BAND RASTER)")
print("=" * 80)

for label, path in [("Census raster", CENSUS_RASTER),
                     ("Heat raster", HEAT_RASTER),
                     ("Input file", INPUT_FILE)]:
    exists = Path(path).exists()
    print(f"  {'✓' if exists else '❌'} {label}: {Path(path).name}")
    if not exists:
        raise FileNotFoundError(path)

start_time = datetime.now()

# ── Raster diagnostics ─────────────────────────────────────────────────────
print("\nCensus raster diagnostics (original)...")
with rasterio.open(CENSUS_RASTER) as src:
    print(f"  CRS      : {src.crs}")
    print(f"  Shape    : {src.height} × {src.width}")
    print(f"  Res (m)  : {src.res}")
    print(f"  Nodata   : {src.nodata}")
    sample = src.read(1)
    valid  = sample[sample > 0]
    print(f"  Value range (>0): {valid.min():.1f} – {valid.max():.1f}  "
          f"({len(valid):,} populated pixels)")

print("\nHeat raster diagnostics (original)...")
with rasterio.open(HEAT_RASTER) as src:
    print(f"  CRS      : {src.crs}")
    print(f"  Shape    : {src.height} × {src.width}")
    print(f"  Bands    : {src.count}  (expected: {len(HEAT_CATEGORIES)})")
    print(f"  Res (m)  : {src.res}")
    print(f"  Nodata   : {src.nodata}")
    if src.count != len(HEAT_CATEGORIES):
        raise ValueError(
            f"Heat raster has {src.count} bands but HEAT_CATEGORIES defines "
            f"{len(HEAT_CATEGORIES)} categories — check band order / raster file."
        )
    for band_idx, cat_name in HEAT_CATEGORIES.items():
        band_sample = src.read(band_idx)
        valid_band = band_sample[band_sample > 0]
        if len(valid_band) > 0:
            print(f"    Band {band_idx} ({cat_name}): "
                  f"range {valid_band.min():.1f}–{valid_band.max():.1f}, "
                  f"{len(valid_band):,} non-zero pixels")
        else:
            print(f"    Band {band_idx} ({cat_name}): no non-zero pixels found")

# ── Load vector file ───────────────────────────────────────────────────────
print("\nLoading input file...")
gdf = gpd.read_file(INPUT_FILE)
print(f"  → {len(gdf)} total rows, {gdf['nhda_id'].nunique()} unique nhda_ids")
print(f"  → type values: {gdf['type'].unique()}")
print(f"  → CRS: {gdf.crs}")

if gdf.crs is None or not gdf.crs.axis_info or gdf.crs.axis_info[0].unit_name not in ("metre", "meter"):
    print("  ⚠️  Warning: could not confirm vector CRS units are metres — "
          "area_ha / population_per_ha may be wrong if the CRS is not projected in metres.")

# ── Reproject rasters if CRS does not match ─────────────────────────────────
print("\nChecking raster CRS alignment...")
census_path, census_nodata = reproject_raster_if_needed(CENSUS_RASTER, gdf.crs, label="Census raster")
heat_path,   heat_nodata   = reproject_raster_if_needed(HEAT_RASTER,   gdf.crs, label="Heat raster")

# ── Step 1: area_ha ─────────────────────────────────────────────────────────
print("\nStep 1: Computing polygon area (ha)...")
gdf = gdf.reset_index(drop=True)
gdf['area_ha'] = gdf.geometry.area / 10_000.0
print(f"  ✓ area_ha computed  |  mean: {gdf['area_ha'].mean():.2f} ha  "
      f"|  min: {gdf['area_ha'].min():.2f}  |  max: {gdf['area_ha'].max():.2f}")

# ── Step 2 + 3: population, population_per_ha, 9 heat columns, dominant type ─
print(f"\nStep 2/3: Computing population, population_per_ha, per-category heat "
      f"sums (9 columns) and dominant heat type for all {len(gdf)} rows...\n")

gdf['population']         = np.nan
gdf['population_per_ha']  = np.nan

heat_col_names = [f'heat_{cat_name}' for cat_name in HEAT_CATEGORIES.values()]
for col in heat_col_names:
    gdf[col] = np.nan

gdf['dominant_heat_type'] = None

pop_error_count  = 0
heat_error_count = 0

for idx, row in gdf.iterrows():
    if idx % 50 == 0:
        elapsed = (datetime.now() - start_time).seconds
        filled_so_far = gdf['population'].notna().sum()
        print(f"  {idx + 1:>5}/{len(gdf)}  |  filled: {filled_so_far}  |  elapsed: {elapsed}s")

    # -- population --
    pop = calculate_population_for_polygon(
        row.geometry, census_path, census_nodata,
        verbose=(pop_error_count < 5)
    )
    if pop is None:
        pop_error_count += 1
    else:
        gdf.at[idx, 'population'] = pop
        area_ha = row['area_ha']
        if area_ha and area_ha > 0:
            gdf.at[idx, 'population_per_ha'] = round(pop / area_ha, 4)

    # -- heating type breakdown (9 bands) --
    band_sums = calculate_building_counts_by_band(
        row.geometry, heat_path, heat_nodata, HEAT_CATEGORIES,
        verbose=(heat_error_count < 5)
    )
    if band_sums is None:
        heat_error_count += 1
    else:
        for cat_name, value in band_sums.items():
            gdf.at[idx, f'heat_{cat_name}'] = value

        dominant = get_dominant_heat_type(band_sums, HEAT_CATEGORIES)
        gdf.at[idx, 'dominant_heat_type'] = dominant

filled_pop  = gdf['population'].notna().sum()
filled_heat = gdf['dominant_heat_type'].notna().sum()

print(f"\n  ✓ population filled: {filled_pop} / {len(gdf)}  |  errors/None: {pop_error_count}"
      f"  |  mean pop/ha: {gdf['population_per_ha'].mean():.2f}")
print(f"  ✓ dominant_heat_type filled: {filled_heat} / {len(gdf)}  |  errors/None: {heat_error_count}")

if filled_pop == 0:
    print("\n  ❌ FATAL: 0 rows filled for population — check raster extent vs. vector extent!")
    with rasterio.open(census_path) as r:
        print(f"     Census raster bounds: {r.bounds}")
    print(f"     Vector bounds: {gdf.total_bounds}")
    raise RuntimeError("No population values could be extracted. See diagnostics above.")

if filled_heat == 0:
    print("\n  ⚠️  WARNING: 0 rows got a dominant heat type — check heat raster extent vs. vector extent!")
    with rasterio.open(heat_path) as r:
        print(f"     Heat raster bounds: {r.bounds}")
    print(f"     Vector bounds: {gdf.total_bounds}")

# ── Step 4: Cross-join partner values by nhda_id ────────────────────────────
print("\nStep 4: Adding partner population / density / heat-breakdown values...")

carry_cols = ['nhda_id', 'population', 'population_per_ha', 'dominant_heat_type'] + heat_col_names

nhda_rename = {
    'population':         'nhda_population',
    'population_per_ha':  'nhda_population_per_ha',
    'dominant_heat_type': 'nhda_dominant_heat_type',
}
nhda_rename.update({col: f'nhda_{col}' for col in heat_col_names})

ra_rename = {
    'population':         'ra_population',
    'population_per_ha':  'ra_population_per_ha',
    'dominant_heat_type': 'ra_dominant_heat_type',
}
ra_rename.update({col: f'ra_{col}' for col in heat_col_names})

nhda_lookup = gdf[gdf['type'] == 'NHDA'][carry_cols].rename(columns=nhda_rename)
ra_lookup   = gdf[gdf['type'] == 'RA'][carry_cols].rename(columns=ra_rename)

result = gdf.merge(nhda_lookup, on='nhda_id', how='left')
result = result.merge(ra_lookup,   on='nhda_id', how='left')

# drop the intermediate per-row columns — only partner columns are kept
result = result.drop(columns=['population', 'population_per_ha', 'dominant_heat_type'] + heat_col_names)

# signed differences: positive → NHDA has more residents / higher density
result['pop_diff_nhda_ra']        = result['nhda_population'] - result['ra_population']
result['pop_diff_per_ha_nhda_ra'] = result['nhda_population_per_ha'] - result['ra_population_per_ha']


def _heat_types_match(row):
    a, b = row['nhda_dominant_heat_type'], row['ra_dominant_heat_type']
    if pd.isna(a) or pd.isna(b):
        return None
    set_a = {value.strip() for value in str(a).split(";") if value.strip()}
    set_b = {value.strip() for value in str(b).split(";") if value.strip()}
    return len(set_a & set_b) > 0


result['heat_type_match'] = result.apply(_heat_types_match, axis=1)

result = gpd.GeoDataFrame(result, crs=gdf.crs)

# ── Summary ────────────────────────────────────────────────────────────────
nhda_rows = result[result['type'] == 'NHDA']
ra_rows   = result[result['type'] == 'RA']

print(f"\n{'=' * 80}")
print("SUMMARY")
print(f"{'=' * 80}")
print(f"  Output rows : {len(result)}  (NHDA: {len(nhda_rows)}, RA: {len(ra_rows)})")

print(f"\n  nhda_population_per_ha filled : {result['nhda_population_per_ha'].notna().sum()}"
      f" ({result['nhda_population_per_ha'].notna().mean() * 100:.1f} %)")
print(f"  ra_population_per_ha  filled  : {result['ra_population_per_ha'].notna().sum()}"
      f" ({result['ra_population_per_ha'].notna().mean() * 100:.1f} %)")

print(f"\nNHDA rows — nhda_population_per_ha statistics:")
print(nhda_rows['nhda_population_per_ha'].describe().round(2).to_string())
print(f"\nRA rows — ra_population_per_ha statistics:")
print(ra_rows['ra_population_per_ha'].describe().round(2).to_string())

print(f"\npop_diff_per_ha_nhda_ra (all rows):")
print(result['pop_diff_per_ha_nhda_ra'].describe().round(2).to_string())
print(f"  NHDA denser : {(result['pop_diff_per_ha_nhda_ra'] > 0).sum()} rows")
print(f"  RA denser   : {(result['pop_diff_per_ha_nhda_ra'] < 0).sum()} rows")

print(f"\nDominant heating type — most common (NHDA rows):")
print(nhda_rows['nhda_dominant_heat_type'].value_counts().head(10).to_string())
print(f"\nDominant heating type — most common (RA rows):")
print(ra_rows['ra_dominant_heat_type'].value_counts().head(10).to_string())

match_counts = result['heat_type_match'].value_counts(dropna=False)
print(f"\nheat_type_match (NHDA vs. RA share ≥1 dominant category):")
print(match_counts.to_string())

# ── Save ───────────────────────────────────────────────────────────────────
output_path = Path(OUTPUT_FILE)
output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_file(OUTPUT_FILE, driver='GPKG')

print(f"\n✓ Done in {datetime.now() - start_time}")
print(f"  Output: {OUTPUT_FILE}  ({output_path.stat().st_size / 1024 / 1024:.1f} MB)")
print("=" * 80 + "\n")

In [ ]:
"""
Dominant heating type per Bavarian district for NHDAs and reference areas.

This script is aligned with the output produced by "Pasted code(80).py".
That script stores the polygon-level heating results in these columns:

    nhda_dominant_heat_type
    ra_dominant_heat_type

For each map, polygons are assigned to a district using representative points.
The polygon-level dominant heating categories are then counted per district.
If a polygon contains several dominant categories separated by "; ", only the
first category in the stored list is used. If several categories have the same
highest district-level count, the first category in CATEGORY_ORDER is used.

Two maps are exported: one for NHDAs and one for reference areas (RAs).
"""

import math
from pathlib import Path

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

try:
    from matplotlib_map_utils import north_arrow, scale_bar
    HAS_MMU = True
except Exception:
    HAS_MMU = False


# =============================================================================
# PLOT SETTINGS
# =============================================================================

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 13,
    "axes.titlesize": 22,
    "axes.labelsize": 13,
    "legend.fontsize": 11,
    "legend.title_fontsize": 12,
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
})


# =============================================================================
# PATHS AND CONFIGURATION
# =============================================================================

PATH_LANDKREISE = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
LAYER_LANDKREISE = "v_vg250_krs"

PATH_HEAT = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure_Differences_Census.gpkg"

OUTPUT_DIR = Path(r"C:\Users\agz90fk\Documents\EO4CAM\07_Abbildungen\Masterarbeit\heating_type_maps")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COL_ARS = "Regionalschlüssel_ARS"
BAYERN_PREFIX = "09"
TARGET_CRS = "EPSG:25832"
CRS_NOTE = "CRS: EPSG:25832 - ETRS89 / UTM zone 32N"
GRID_STEP_M = 50_000
MAP_PADDING_M = 10_000

NHDA_HEAT_COLUMN = "nhda_dominant_heat_type"
RA_HEAT_COLUMN = "ra_dominant_heat_type"


# =============================================================================
# HEATING CATEGORIES AND COLORS
# Names must match the output strings from code (80).
# =============================================================================

HEAT_LABELS = {
    "Gas": "Gas",
    "Heating_oil": "Heating oil",
    "Wood_pallets": "Wood / pellets",
    "Biomass_Non_wood_Biogas": "Biomass / biogas",
    "Solar_geothermal_heat_pump": "Solar / geothermal / heat pump",
    "Electric_heating_no_heat_pumps": "Electric heating (no heat pump)",
    "Coal": "Coal",
    "District_Heating": "District heating",
    "No_heating": "No heating",
}

HEAT_COLORS = {
    "Gas": "#E85C1A",
    "Heating_oil": "#7B3F00",
    "Wood_pallets": "#4A7C2F",
    "Biomass_Non_wood_Biogas": "#91C46C",
    "Solar_geothermal_heat_pump": "#F5C400",
    "Electric_heating_no_heat_pumps": "#3A86C8",
    "Coal": "#6B6B6B",
    "District_Heating": "#B3176E",
    "No_heating": "#C8C8C8",
}

CATEGORY_ORDER = list(HEAT_LABELS.keys())
NO_DATA_COLOR = "#EBEBEB"


# =============================================================================
# MAP HELPERS
# =============================================================================

def add_matplotlib_grid(ax, bounds, step=GRID_STEP_M):
    minx, miny, maxx, maxy = bounds

    first_x = math.ceil(minx / step) * step
    last_x = math.floor(maxx / step) * step
    first_y = math.ceil(miny / step) * step
    last_y = math.floor(maxy / step) * step

    xs = list(np.arange(first_x, last_x + step, step)) if first_x <= last_x else []
    ys = list(np.arange(first_y, last_y + step, step)) if first_y <= last_y else []

    ax.set_xticks(xs)
    ax.set_yticks(ys)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(round(x / 1000))}"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{int(round(y / 1000))}"))

    if xs and ys:
        xp, yp = zip(*[(x, y) for x in xs for y in ys])
        ax.scatter(
            xp,
            yp,
            marker="+",
            s=30,
            linewidths=1.0,
            color="#8a8a8a",
            alpha=0.95,
            zorder=4,
            clip_on=True,
        )

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=11,
        length=0,
        colors="#9B9999",
    )


def add_north_arrow(ax):
    if HAS_MMU:
        north_arrow(
            ax=ax,
            location="upper right",
            size="md",
            rotation={"degrees": 0},
            aob={
                "bbox_to_anchor": (0.985, 0.985),
                "bbox_transform": ax.transAxes,
                "pad": 0.06,
                "borderpad": 0.06,
                "facecolor": "none",
                "edgecolor": "none",
                "frameon": False,
            },
        )
        return

    ax.annotate(
        "",
        xy=(0.945, 0.960),
        xytext=(0.945, 0.880),
        xycoords="axes fraction",
        textcoords="axes fraction",
        arrowprops={
            "arrowstyle": "-|>",
            "color": "#222222",
            "linewidth": 1.4,
            "shrinkA": 0,
            "shrinkB": 0,
        },
        zorder=10,
    )
    ax.text(
        0.945,
        0.972,
        "N",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold",
        color="#222222",
        zorder=10,
    )


def add_scale_bar(ax):
    if HAS_MMU:
        scale_bar(
            ax=ax,
            location="lower right",
            size="xs",
            style="ticks",
            bar={
                "projection": TARGET_CRS,
                "unit": "km",
                "max": 50,
                "major_div": 2,
                "minor_div": 1,
                "minor_type": "none",
            },
            labels={
                "labels": ["0", "25", "50"],
                "style": "major",
                "loc": "below",
                "fontsize": 9,
            },
            units={"loc": "text", "label": "km"},
            text={
                "fontfamily": "sans-serif",
                "fontsize": 9,
                "textcolor": "#222222",
            },
            aob={
                "pad": 0.0,
                "borderpad": 1.0,
                "facecolor": "none",
                "edgecolor": "none",
                "frameon": False,
            },
        )
        return

    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    sx, sy = x1 - x0, y1 - y0
    bar_len = 50_000
    seg_len = 25_000
    xb = x1 - sx * 0.35
    yb = y0 + sy * 0.060

    ax.plot([xb, xb + bar_len], [yb, yb], color="#222222", linewidth=1.3, zorder=8)
    tick_height = sy * 0.006

    for x in [xb, xb + seg_len, xb + bar_len]:
        ax.plot(
            [x, x],
            [yb - tick_height, yb + tick_height],
            color="#222222",
            linewidth=1.0,
            zorder=8,
        )

    text_y = yb + sy * 0.011
    for x, label in [(xb, "0"), (xb + seg_len, "25"), (xb + bar_len, "50 km")]:
        ax.text(x, text_y, label, ha="center", va="bottom", fontsize=9, color="#222222")


def add_scientific_frame(ax, gdf_base):
    ax.set_facecolor("#f1f1f1")
    minx, miny, maxx, maxy = gdf_base.total_bounds
    bounds = (
        minx - MAP_PADDING_M,
        miny - MAP_PADDING_M,
        maxx + MAP_PADDING_M,
        maxy + MAP_PADDING_M,
    )

    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.margins(0)
    add_matplotlib_grid(ax, bounds)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_autoscale_on(False)

    ax.set_xlabel("Easting (km) – UTM 32N", fontsize=12, color="#555555")
    ax.set_ylabel("Northing (km) – UTM 32N", fontsize=12, color="#555555")
    ax.text(
        0.01,
        0.01,
        CRS_NOTE,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=11,
        color="#555555",
    )

    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("#636262")
        ax.spines[side].set_linewidth(0.8)


def add_legend_bottom_right(ax, handles):
    ax.legend(
        handles=handles,
        title="Dominant\nheating type",
        loc="lower left",
        bbox_to_anchor=(1.01, 0.02),
        borderaxespad=0.0,
        frameon=False,
        fontsize=13,
        title_fontsize=14,
        handlelength=2.0,
        labelspacing=0.6,
        borderpad=0.0,
    )


# =============================================================================
# DATA HELPERS
# =============================================================================

def validate_columns(gdf):
    required = {
        "type",
        "geometry",
        NHDA_HEAT_COLUMN,
        RA_HEAT_COLUMN,
    }
    missing = sorted(required.difference(gdf.columns))
    if missing:
        raise ValueError(
            "The heating GeoPackage is missing the following required columns: "
            + ", ".join(missing)
            + "\nAvailable columns are:\n"
            + ", ".join(gdf.columns)
        )


def parse_first_heat_category(value):
    """
    Return the first valid heating category stored in a polygon-level value.

    Examples
    --------
    'Gas' -> 'Gas'
    'Gas; Heating_oil' -> 'Gas'

    The order written by code (80) is retained. Unknown or missing values
    return None.
    """
    if pd.isna(value):
        return None

    parts = [part.strip() for part in str(value).split(";") if part.strip()]
    for category in parts:
        if category in HEAT_LABELS:
            return category
    return None


def dominant_heating_per_district(gdf_polygons, gdf_districts, heat_column):
    """
    Determine the most frequent polygon-level heating category per district.

    For polygon values containing several tied categories, only the first
    category in the list is counted. For a district-level tie, the first
    category in CATEGORY_ORDER is selected.
    """
    if gdf_polygons.empty:
        return pd.Series(dtype="object")

    points = gdf_polygons[[heat_column, "geometry"]].copy()
    points["geometry"] = points.geometry.representative_point()

    joined = gpd.sjoin(
        points,
        gdf_districts[["geometry"]],
        how="left",
        predicate="within",
    )

    unmatched = joined["index_right"].isna().sum()
    if unmatched:
        print(f"  Warning: {unmatched} polygons could not be assigned to a district.")

    joined["heat_category"] = joined[heat_column].apply(parse_first_heat_category)

    unknown_mask = joined[heat_column].notna() & joined["heat_category"].isna()
    if unknown_mask.any():
        print("  Warning: ignored unknown heating labels:")
        for value in sorted(joined.loc[unknown_mask, heat_column].astype(str).unique()):
            print(f"    - {value}")

    valid = joined.dropna(subset=["index_right", "heat_category"]).copy()
    if valid.empty:
        return pd.Series(dtype="object")

    valid["district_index"] = valid["index_right"].astype(int)

    counts = (
        valid.groupby(["district_index", "heat_category"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=CATEGORY_ORDER, fill_value=0)
    )

    def district_winner(row):
        max_value = row.max()
        if max_value <= 0:
            return None

        # CATEGORY_ORDER defines the priority when district counts are tied.
        return next(
            category for category in CATEGORY_ORDER
            if row[category] == max_value
        )

    return counts.apply(district_winner, axis=1)


# =============================================================================
# PLOTTING
# =============================================================================

def plot_heating_map(dominant_series, gdf_districts, title, out_path):
    gdf_plot = gdf_districts.copy()

    # Explicit mapping by district index avoids accidental index alignment issues.
    gdf_plot["dominant_heat"] = gdf_plot.index.to_series().map(dominant_series)

    fig, ax = plt.subplots(figsize=(8.6, 9.4))
    fig.subplots_adjust(right=0.78, left=0.07, top=0.92, bottom=0.08)

    no_data = gdf_plot["dominant_heat"].isna()
    if no_data.any():
        gdf_plot[no_data].plot(
            ax=ax,
            color=NO_DATA_COLOR,
            edgecolor="#b7b7b7",
            linewidth=0.4,
            zorder=2,
        )

    used_values = []
    for category in CATEGORY_ORDER:
        mask = gdf_plot["dominant_heat"] == category
        if not mask.any():
            continue

        color = HEAT_COLORS[category]
        hatch = None

        gdf_plot[mask].plot(
            ax=ax,
            color=color,
            edgecolor="#555555",
            linewidth=0.4,
            hatch=hatch,
            zorder=2,
        )
        used_values.append(category)

    add_scientific_frame(ax, gdf_plot)
    add_north_arrow(ax)
    add_scale_bar(ax)

    legend_handles = []
    for category in CATEGORY_ORDER:
        if category not in used_values:
            continue

        label = HEAT_LABELS[category]
        color = HEAT_COLORS[category]
        hatch = None

        legend_handles.append(
            mpatches.Patch(
                facecolor=color,
                edgecolor="#555555",
                linewidth=0.5,
                hatch=hatch,
                label=label,
            )
        )

    legend_handles.append(
        mpatches.Patch(
            facecolor=NO_DATA_COLOR,
            edgecolor="#b7b7b7",
            linewidth=0.5,
            label="No data",
        )
    )
    add_legend_bottom_right(ax, legend_handles)

    ax.set_title(title, fontsize=22, fontweight="bold", pad=10)
    fig.savefig(out_path, dpi=300, bbox_inches="tight", format="jpg")
    plt.close(fig)
    print(f"  Saved: {out_path}")


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("Loading Bavarian districts...")
    gdf_lk = gpd.read_file(PATH_LANDKREISE, layer=LAYER_LANDKREISE)

    ars_col = COL_ARS if COL_ARS in gdf_lk.columns else next(
        (column for column in gdf_lk.columns if "ARS" in column.upper()),
        None,
    )
    if ars_col is None:
        raise ValueError("No ARS column was found in the district dataset.")

    # Normalize administrative keys without padding them before the state test.
    # Examples that must all be recognized as Bavaria:
    #   "09162", 9162, "091620000000", 91620000000, "09162.0"
    ars_raw = (
        gdf_lk[ars_col]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
    )

    # When a key was read numerically, Germany's leading zero is lost.
    # Thus Bavarian keys begin either with "09" (zero retained) or "9"
    # (zero lost). Do not use zfill here because the source may contain
    # either a 5-digit AGS or a 12-digit ARS.
    is_bavaria = ars_raw.str.startswith("09") | ars_raw.str.startswith("9")
    gdf_lk = gdf_lk[is_bavaria].copy()

    if gdf_lk.empty:
        sample_values = ars_raw.dropna().drop_duplicates().head(10).tolist()
        raise ValueError(
            f"No Bavarian districts were selected from column '{ars_col}'. "
            f"Example normalized values: {sample_values}"
        )

    if gdf_lk.crs is None:
        raise ValueError("The district dataset has no CRS.")
    if gdf_lk.crs.to_epsg() != 25832:
        gdf_lk = gdf_lk.to_crs(TARGET_CRS)

    gdf_lk = gdf_lk.reset_index(drop=True)
    bayern_geometry = gdf_lk.geometry.union_all()

    print(f"  Districts selected: {len(gdf_lk)}")

    print("Loading heating data...")
    gdf_heat = gpd.read_file(PATH_HEAT)
    validate_columns(gdf_heat)

    if gdf_heat.crs is None:
        raise ValueError("The heating GeoPackage has no CRS.")
    if gdf_heat.crs != gdf_lk.crs:
        gdf_heat = gdf_heat.to_crs(gdf_lk.crs)

    gdf_heat = gdf_heat[
        gdf_heat.geometry.notna()
        & ~gdf_heat.geometry.is_empty
        & gdf_heat.geometry.intersects(bayern_geometry)
    ].copy()

    type_normalized = gdf_heat["type"].astype(str).str.strip().str.upper()
    nhda = gdf_heat[type_normalized == "NHDA"].copy()
    ra = gdf_heat[type_normalized == "RA"].copy()

    print(f"  Polygons loaded: {len(gdf_heat)}")
    print(f"  NHDA polygons:    {len(nhda)}")
    print(f"  RA polygons:      {len(ra)}")

    if nhda.empty or ra.empty:
        raise ValueError(
            "No NHDA or RA rows were found. Check the values in the 'type' column."
        )

    print("Computing district-level dominant heating types...")
    nhda_dominant = dominant_heating_per_district(
        nhda,
        gdf_lk,
        NHDA_HEAT_COLUMN,
    )
    ra_dominant = dominant_heating_per_district(
        ra,
        gdf_lk,
        RA_HEAT_COLUMN,
    )

    print("\nNHDA district results:")
    print(nhda_dominant.value_counts(dropna=False).to_string())
    print("\nRA district results:")
    print(ra_dominant.value_counts(dropna=False).to_string())

    print("\nCreating NHDA map...")
    plot_heating_map(
        nhda_dominant,
        gdf_lk,
        title="Dominant Heating Type – NHDA",
        out_path=OUTPUT_DIR / "heating_dominant_NHDA.jpg",
    )

    print("Creating RA map...")
    plot_heating_map(
        ra_dominant,
        gdf_lk,
        title="Dominant Heating Type – RA",
        out_path=OUTPUT_DIR / "heating_dominant_RA.jpg",
    )

    print("\nDone. Both maps were exported.")


if __name__ == "__main__":
    main()
